In [31]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
# Load dataset 
df=pd.read_csv("dataset.csv")

In [33]:
print("Dataset shape:",df.shape)
print("\nCOlumns:")
df.head()

Dataset shape: (8790, 10)

COlumns:


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [34]:
# EDA
print("Row and coluns",df.shape)

Row and coluns (8790, 10)


In [35]:
# Statistical information
df.describe()

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [37]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'],
      dtype='object')

In [38]:
# Step 1: Extract relevant features

features=['type','title','director','country','listed_in']
df=df[features].copy()

print("Relevant features extracted")
print(df.head())

Relevant features extracted
      type                             title         director        country  \
0    Movie              Dick Johnson Is Dead  Kirsten Johnson  United States   
1  TV Show                         Ganglands  Julien Leclercq         France   
2  TV Show                     Midnight Mass    Mike Flanagan  United States   
3    Movie  Confessions of an Invisible Girl    Bruno Garotti         Brazil   
4    Movie                           Sankofa     Haile Gerima  United States   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dramas, TV Horror, TV Mysteries  
3                 Children & Family Movies, Comedies  
4   Dramas, Independent Movies, International Movies  


In [39]:
# handle missing values
for col in features:
    df[col]=df[col].fillna(" ")
print(df[features].isnull().sum())

type         0
title        0
director     0
country      0
listed_in    0
dtype: int64


In [40]:
# Step 2 :Text preprocessing
def clean_text(text):
    return str(text).lower().strip().replace(","," ")
df["combined_features"]=(df["type"].apply(clean_text)+ " " +
                          df["title"].apply(clean_text)+" "+
                          df["director"].apply(clean_text)+ " " +
                          df["country"].apply(clean_text)+ " " +
                          df["listed_in"].apply(clean_text))

df[["title","combined_features"]].head()

,title,combined_features
0,Dick Johnson Is Dead,movie dick johnson is dead kirsten johnson uni...
1,Ganglands,tv show ganglands julien leclercq france crime...
2,Midnight Mass,tv show midnight mass mike flanagan united sta...
3,Confessions of an Invisible Girl,movie confessions of an invisible girl bruno g...
4,Sankofa,movie sankofa haile gerima united states drama...


In [41]:
# Step 3: Calculate content similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["combined_features"])

similarity = cosine_similarity(tfidf_matrix)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Similarity Matrix Shape:", similarity.shape)

TF-IDF Matrix Shape: (8790, 14325)
Similarity Matrix Shape: (8790, 8790)


In [42]:
# Step 4: Generates recommendations
def recommend_movies(movie_name, top_n=5):
    matches = df[df["title"].str.lower() == movie_name.lower()]

    if matches.empty:
        print("Movie not found. Please check the title.")
        return

    idx = matches.index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    scores = [
        item for item in scores
        if item[0] != idx
    ][:top_n]

    recommended_indices = [item[0] for item in scores]

    result = df.loc[recommended_indices, ["title", "type", "listed_in"]].copy()
    result["similarity_score"] = [round(item[1], 3) for item in scores]

    return result.reset_index(drop=True)

In [43]:
recommend_movies("The Conjuring", 5)

,title,type,listed_in,similarity_score
0,The Conjuring 2,Movie,Horror Movies,0.910
1,Insidious,Movie,"Horror Movies, Thrillers",0.647
2,Conjuring Spirit,Movie,"Horror Movies, International Movies, Thrillers",0.381
3,Final Destination,Movie,"Cult Movies, Horror Movies, Thrillers",0.316
4,Final Destination 3,Movie,Horror Movies,0.280


In [44]:
print(df["title"].head(20).to_list())

['Dick Johnson Is Dead', 'Ganglands', 'Midnight Mass', 'Confessions of an Invisible Girl', 'Sankofa', 'The Great British Baking Show', 'The Starling', 'Motu Patlu in the Game of Zones', 'Je Suis Karl', 'Motu Patlu in Wonderland', 'Motu Patlu: Deep Sea Adventure', 'Motu Patlu: Mission Moon', '99 Songs (Tamil)', 'Bridgerton - The Afterparty', 'Bling Empire - The Afterparty', 'Cobra Kai - The Afterparty', 'The Circle - The Afterparty', 'Jailbirds New Orleans', 'Crime Stories: India Detectives', 'True: Winter Wishes']


In [45]:
# Step 5 : Evalute recommendation quality
movie_name=df["title"].dropna().iloc[0]
result=recommend_movies(movie_name,5)
print("Selected Movie:",movie_name)
print("\nRecommended Movies:")
print(result)

Selected Movie: Dick Johnson Is Dead

Recommended Movies:
           title   type                                          listed_in  \
0       S.W.A.T.  Movie                                 Action & Adventure   
1  Triple Threat  Movie           Action & Adventure, International Movies   
2    Nowhere Boy  Movie                                             Dramas   
3          Brick  Movie                      Independent Movies, Thrillers   
4           Home  Movie  Children & Family Movies, Comedies, Sci-Fi & F...   

   similarity_score  
0             0.420  
1             0.366  
2             0.333  
3             0.323  
4             0.321  
